In [17]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [25]:
from langchain.chat_models import init_chat_model
import os
from langchain_openai import ChatOpenAI
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 创建智能体
llm = ChatOpenAI(
    model="qwen3.7-plus",
    base_url=base_url,
    api_key=api_key,
    extra_body={"enable_thinking": False},  # 关闭思考模式
)

# 多模态模型（直接使用已创建的模型实例）
multimodal_model = llm

# 定义工具


In [26]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理

这里我们采用sqllite

In [27]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [28]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [31]:
from langchain_core.messages import HumanMessage

# 图片链接
image_url = "https://pic.52112.com/180606/JPG-180606_150/KFC5muPJoj_small.jpg"

multimodal_message = HumanMessage([
    {"type":"text","text":"帮我看看能做什么？"},
    {"type":"image_url","image_url": {"url": image_url}}
])

config = {"configurable": {"thread_id": "2"}}

In [32]:
print(multimodal_message.content)
response = agent.invoke({"messages": [multimodal_message]}, config)

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image_url', 'image_url': {'url': 'https://pic.52112.com/180606/JPG-180606_150/KFC5muPJoj_small.jpg'}}]


In [33]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image_url', 'image_url': {'url': 'https://pic.52112.com/180606/JPG-180606_150/KFC5muPJoj_small.jpg'}}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_3a122eb7b6994c9ea520c107)
 Call ID: call_3a122eb7b6994c9ea520c107
  Args:
    query: 三文鱼 西兰花 牛油果 鸡蛋 彩椒 菠菜 鸡胸肉 牛肉 食谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "三文鱼 西兰花 牛油果 鸡蛋 彩椒 菠菜 鸡胸肉 牛肉 食谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://m.xiachufang.com/recipe/106825427", "title": "【步骤图】早午餐的做法 - 下厨房", "content": "芝士辣白菜鸡肉卷饼，蔬菜燕麦蛋糕，菌菇西兰花，生菜，无花果\n\n步骤 44\n\n纯奶手撕吐司的做法 步骤1\n\n烤南瓜西葫芦鸡胸肉，煎蛋，紫薯，口蘑，麻酱菠菜荷兰豆，火龙果\n\n步骤 45\n\n纯奶手撕吐司的做法 步骤1\n\n蔬菜鸡肉饭团，口蘑菠菜烘蛋，生菜，牛油果，黄桃\n\n步骤 46\n\n纯奶手撕吐司的做法 步骤1\n\n牛肉汉堡，煎蛋，黄瓜，橙子\n\n步骤 47\

In [34]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

In [35]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

太棒了！“三文鱼西兰花烘蛋”确实是一道颜值与营养并存的懒人神菜。为了让您一次成功，我为您整理了超详细的实操指南：

### 🍳 三文鱼西兰花烘蛋 · 详细制作指南

#### 1. 食材准备（1人份）
*   **核心蛋白**：三文鱼 1块（约80-100g，切丁）、鸡蛋 2个、牛奶或无糖豆奶 30ml
*   **蔬菜搭配**：西兰花 4-5小朵、小番茄 3-4颗（对半切）
*   **调味点睛**：盐 少许、黑胡椒碎 适量、橄榄油 几滴、帕玛森芝士碎 少许（可选，增加奶香）

#### 2. 关键预处理步骤
*   **三文鱼去腥增香**：切好的三文鱼丁用少许盐和黑胡椒抓匀，腌制5分钟。这一步不仅能去腥，还能让鱼肉在烘烤时更入味。
*   **西兰花焯水技巧**：锅中水烧开，加一小勺盐和几滴油（保持翠绿的关键），放入西兰花焯水1-2分钟至断生即可捞出沥干。**注意不要煮太久**，否则烘烤后会软烂失去口感。
*   **蛋液黄金比例**：鸡蛋打散后加入牛奶/豆奶搅匀，过筛一次能让成品更细腻蓬松。此时可加入少许芝士碎，烘烤后会形成诱人的焦香表层。

#### 3. 组合与烘烤
*   **铺底**：烤盘底部刷一层薄油防粘，先铺上焯好的西兰花，再放上小番茄和腌好的三文鱼丁。
*   **倒蛋液**：将调好的蛋液均匀淋在食材上，确保覆盖所有缝隙。
*   **温度时间**：烤箱提前预热至200°C，放入中层烤12-15分钟。观察状态：**表面金黄、蛋液完全凝固、三文鱼边缘微焦**即为最佳出炉时机。若没有烤箱，可用平底锅小火加盖焖煎10分钟，最后开盖收干水分。

#### 4. 升级小技巧
*   **更清爽版**：将牛奶换成无糖燕麦奶或椰浆，适合乳糖不耐受人群，且带有独特植物香气。
*   **更高纤版**：在铺底时加入一把新鲜菠菜叶或洋葱丁，不仅颜色更丰富，还能额外补充膳食纤维。
*   **口感加分项**：出炉后撒少许坚果碎（如杏仁片、碧根果碎），利用余温激发出坚果油脂香，与三文鱼的油脂感形成绝妙呼应。

#### 5. 营养亮点解析
这道菜的蛋白质来源实现了“动物+植物”双重互补：三文鱼提供Omega-

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [7]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

根据您提供的食材（土豆、猪肉、胡萝卜、番茄、青椒、茄子、大蒜、胡椒粒），我为您整理了一份营养均衡、制作简单的菜谱推荐清单。这些菜谱结合了您现有的食材，兼顾美味与健康，适合日常家庭烹饪。

---

### 🍽️ **推荐菜谱排名（按营养与制作难度综合评分）**

#### **1. 地三鲜（土豆、茄子、青椒）——经典家常菜（推荐指数：★★★★★）**
- **推荐理由**：这道菜是经典的“地三鲜”，搭配土豆、茄子和青椒，口感丰富，营养均衡。加入猪肉和大蒜，味道更香浓。
- **制作难度**：中等（约30分钟）
- **适合人群**：家庭聚餐、下饭神器
- **做法简述**：
  1. 土豆、茄子、青椒切块，胡萝卜切丝，猪肉切丁。
  2. 热锅加油，先煎土豆至金黄，盛出备用。
  3. 同一锅中炒香蒜末、姜片，加入猪肉丁翻炒至变色。
  4. 加入茄子、青椒、胡萝卜丝，翻炒至软。
  5. 加入生抽、盐、糖调味，最后加入土豆块，翻炒均匀即可。
  6. 可撒上胡椒粒或葱花点缀。

#### **2. 肉末茄子煲（推荐指数：★★★★☆）**
- **推荐理由**：茄子吸汁入味，搭配肉末和番茄，汤汁浓郁，非常适合配米饭。
- **制作难度**：中等（约40分钟）
- **适合人群**：喜欢浓郁口味的人
- **做法简述**：
  1. 茄子切块，用盐腌制10分钟，挤干水分。
  2. 热锅加油，将茄子煎至表面微焦，盛出备用。
  3. 锅中留底油，炒香蒜末、姜片，加入肉末翻炒至变色。
  4. 加入番茄块，炒至出汁，再加入茄子块、胡萝卜丝。
  5. 加入生抽、盐、糖调味，加适量水，小火炖煮15分钟。
  6. 最后撒上胡椒粒，收汁即可。

#### **3. 番茄胡萝卜炒土豆丝（推荐指数：★★★★）**
- **推荐理由**：这道菜色彩鲜艳，口感清爽，适合喜欢清淡口味的人。
- **制作难度**：低（约20分钟）
- **适合人群**：素食者、减脂人群
- **做法简述**：
  1. 土豆切丝，胡萝卜切丝，番茄切块。
  2. 热锅加油，先炒香蒜末，加入土豆丝翻炒至断生。
  3. 加入胡萝卜丝、番茄块，翻炒至番茄出汁。
  4. 加入盐、糖调味，撒上胡椒粒，翻炒均匀即可。

#### **4. 青椒炒肉片配土豆（推荐指数：★★★☆）**
- **推荐理由**：这道

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [7]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-03-05T04:32:04.748794+00:00',
 'id': '1f1184c4-318f-6c21-8005-8152a8ca695c',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='a29981d9-1396-4397-9c42-8a3d4322a691'),
   AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen3-omni-flash', 'model_provider': 'openai'}, id='lc_run--019cbc43-db26-71e0-8385-79361516f7a7', tool_calls=[{'name': 'tavily_search', 'args': {'query': '用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱', 'search_depth': 'advanced'}, 'id': 'call_3971f410f521440fab0430', 'type': 'tool_call'}], invalid_tool_calls=[]),
   ToolMessage(content='{"query": "用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.co

In [71]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='8c0219c3-2c70-46bb-bfa2-400a516e0117'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 2151, 'total_tokens': 2206, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 55}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 262, 'text_tokens': 1889}}, 'model_provider': 'openai', 'model_name': 'qwen3-omni-flash', 'system_fingerprint': None, 'id': 'chatcmpl-a5c8f901-eaa4-9903-96e4-334a9cb802da', 'finish_reason': 'tool_calls', 'logprobs': None} id='l

定义一个便捷的获取会话历史的函数：


In [37]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [39]:
get_messages("2")

[{'role': 'user',
  'content': [{'type': 'text', 'text': '帮我看看能做什么？'},
   {'type': 'image_url',
    'image_url': {'url': 'https://pic.52112.com/180606/JPG-180606_150/KFC5muPJoj_small.jpg'}}]},
 {'role': 'assistant',
  'content': '根据您提供的食材照片，我为您整理了一份“当前可用食材清单”，并基于此检索了多款食谱。经过营养价值和制作难度的综合评估，为您推荐以下三道优选菜品：\n\n### 🥗 1. 三文鱼西兰花烘蛋 (强烈推荐)\n*   **综合得分**：95/100\n*   **推荐理由**：**高蛋白、低脂、懒人友好**。这道菜完美利用了图中的三文鱼、西兰花、小番茄和鸡蛋。制作极其简单（烤箱或平底锅均可），且营养密度极高，富含Omega-3和优质蛋白，饱腹感强且不油腻，非常适合作为健康早午餐或晚餐。\n*   **核心食材**：三文鱼、西兰花、小番茄、鸡蛋、牛奶/豆奶。\n*   **制作难度**：⭐️⭐️ (非常简单)\n\n### 🥩 2. 彩椒牛肉炒西兰花\n*   **综合得分**：88/100\n*   **推荐理由**：**经典快手家常菜**。利用图中的牛肉、黄彩椒和西兰花，是一道色香味俱全的菜肴。彩椒富含维生素C，牛肉补铁，西兰花提供膳食纤维。制作时间短（约15分钟），非常适合下班后快速解决一顿营养均衡的正餐。\n*   **核心食材**：牛肉、黄彩椒、西兰花、蒜末。\n*   **制作难度**：⭐️⭐️⭐️ (中等偏易)\n\n### 🍗 3. 西芹彩椒炒鸡胸肉\n*   **综合得分**：85/100\n*   **推荐理由**：**低脂增肌首选**。结合图中的鸡胸肉、彩椒和西芹（背景可见），这是一道典型的健身餐改良版。鸡胸肉脂肪含量极低，搭配蔬菜口感清爽不柴。虽然鸡胸肉处理不当容易干柴，但通过腌制和快炒技巧可以保持嫩滑，是控制热量摄入的绝佳选择。\n*   **核心食材**：鸡胸肉、彩椒、西芹、洋葱。\n*   **制作难度**：⭐️⭐️⭐️ (需注意火候)\n\n---\n\n### 💡 厨师小贴士\n*   *

# 清空会话历史


In [40]:
checkpointer.delete_thread("1")